[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/11_LangGraph/notebooks/02_human_in_the_loop_and_multi_agent.ipynb)

# Human-in-the-Loop & Multi-Agent Systems

Notebook 11 in Module 10 *talked about* "human-in-the-loop pauses before anything risky" as a
production concern, but never actually built one — there was no mechanism for a Python chain
to stop mid-execution, wait for a person, and continue exactly where it left off. That
mechanism is `interrupt()`, and it's the same checkpointing machinery from Notebook 01's
memory section, used a different way: a checkpoint isn't just "resume after a restart," it's
also "resume after asking a human."

## What you'll build today

1. **`interrupt()` + `Command(resume=...)`** — pause a tool call for human approval
2. **Long-term memory (`Store`)** — facts that persist *across* different conversations, not
   just within one
3. **Multi-agent supervisor** — one router agent handing work to specialists via
   `Command(goto=...)`

## 0. Setup

In [ ]:
%pip install -q langgraph>=0.6 langchain>=1.0 langchain-openai python-dotenv

import warnings, os
warnings.filterwarnings("ignore")
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent.parent / ".env")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. `interrupt()` — pausing for human approval

The scenario: an agent can check the weather freely, but sending an email needs a human to
approve it first. We route tool calls through a `human_approval` node *only* when the
requested tool is the risky one.

`interrupt(payload)` does three things in one call: it (1) freezes graph execution at that
exact point, (2) persists the current state via the checkpointer, and (3) returns `payload`
to whoever called `.invoke()`/`.stream()`, so they can show it to a human.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city (simulated)."""
    return f"{city}: 29C, sunny"


@tool
def send_email(to: str, body: str) -> str:
    """Send an email (simulated)."""
    return f"Email sent to {to}: {body[:50]}..."


risky_tools = {"send_email"}
tools = [get_weather, send_email]
llm_with_tools = llm.bind_tools(tools)


def call_model(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def human_approval(state: MessagesState) -> Command:
    last = state["messages"][-1]
    tool_call = last.tool_calls[0]
    decision = interrupt({
        "reason": "This tool call needs human approval before it runs.",
        "tool": tool_call["name"],
        "args": tool_call["args"],
    })
    if decision.get("approved"):
        return Command(goto="tools")
    rejection = ToolMessage(content="Rejected by human reviewer.", tool_call_id=tool_call["id"])
    return Command(goto="agent", update={"messages": [rejection]})


def route_after_model(state: MessagesState) -> str:
    last = state["messages"][-1]
    if not getattr(last, "tool_calls", None):
        return END
    return "human_approval" if last.tool_calls[0]["name"] in risky_tools else "tools"


hitl_builder = StateGraph(MessagesState)
hitl_builder.add_node("agent", call_model)
hitl_builder.add_node("human_approval", human_approval)
hitl_builder.add_node("tools", ToolNode(tools))
hitl_builder.add_edge(START, "agent")
hitl_builder.add_conditional_edges(
    "agent", route_after_model, {"human_approval": "human_approval", "tools": "tools", END: END}
)
hitl_builder.add_edge("tools", "agent")

hitl_app = hitl_builder.compile(checkpointer=InMemorySaver())

Run it. The graph will stop *inside* `human_approval` and hand back an `__interrupt__` payload
instead of a final answer — that's the pause.

In [ ]:
config = {"configurable": {"thread_id": "approval-demo"}}
result = hitl_app.invoke(
    {"messages": [("user", "Email jane@example.com to confirm tomorrow's 10am meeting.")]},
    config,
)

if "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    print("PAUSED -- waiting for a human:")
    print(" ", payload)
else:
    result["messages"][-1].pretty_print()

Resume it with `Command(resume=...)`, on the **same `thread_id`** — the graph continues from
inside `human_approval`, not from the top.

In [ ]:
approved = hitl_app.invoke(Command(resume={"approved": True}), config)
approved["messages"][-1].pretty_print()

# Try it again on a fresh thread, but reject this time:
config2 = {"configurable": {"thread_id": "approval-demo-2"}}
hitl_app.invoke({"messages": [("user", "Email bob@example.com telling him he's fired.")]}, config2)
rejected = hitl_app.invoke(Command(resume={"approved": False}), config2)
rejected["messages"][-1].pretty_print()

## 2. Long-term memory — facts that outlive one conversation

`checkpointer` (Notebook 01) gives you memory *within* one `thread_id`. A `Store` gives you
memory that any thread can read or write — a real "remember this about the user, forever"
mechanism. Nodes get the store injected automatically when you declare a `store=` keyword-only
parameter.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()


def remember_tone(state: MessagesState, *, store) -> dict:
    text = state["messages"][-1].content
    tone = "casual" if any(w in text.lower() for w in ["hey", "yo", "lol"]) else "formal"
    store.put(("preferences", "user-42"), "tone", {"value": tone})
    return {}


def greet_with_tone(state: MessagesState, *, store) -> dict:
    item = store.get(("preferences", "user-42"), "tone")
    tone = item.value["value"] if item else "neutral"
    greeting = ("Hey! What's up?" if tone == "casual"
                else "Good day. How may I assist you?")
    from langchain_core.messages import AIMessage
    return {"messages": [AIMessage(content=f"[{tone} tone recalled] {greeting}")]}


store_builder = StateGraph(MessagesState)
store_builder.add_node("remember_tone", remember_tone)
store_builder.add_node("greet_with_tone", greet_with_tone)
store_builder.add_edge(START, "remember_tone")
store_builder.add_edge("remember_tone", "greet_with_tone")
store_builder.add_edge("greet_with_tone", END)

store_app = store_builder.compile(store=store)

# Thread 1 today:
store_app.invoke({"messages": [("user", "yo what's good")]}, {"configurable": {"thread_id": "t1"}})
# A completely DIFFERENT thread, days later -- still recalls the preference:
out = store_app.invoke({"messages": [("user", "Hello.")]}, {"configurable": {"thread_id": "t2-days-later"}})
out["messages"][-1].pretty_print()

## 3. Multi-agent supervisor

A supervisor node classifies the request and hands off to a specialist via
`Command(goto=...)`. Each specialist hands control back to the supervisor the same way. This
is the pattern behind the M10 curriculum line "supervisor pattern" — and it's built from
nothing more exotic than the `Command` you already used for the approval handoff above.

In [ ]:
def supervisor(state: MessagesState) -> Command:
    last_user_msg = state["messages"][-1].content
    verdict = llm.invoke(
        "Route this request to exactly one worker: research_agent, math_agent, or FINISH.\n"
        f"Request: {last_user_msg}\nAnswer with one word only."
    ).content.strip().upper()
    if "FINISH" in verdict or len(state["messages"]) > 6:
        return Command(goto=END)
    goto = "research_agent" if "RESEARCH" in verdict else "math_agent"
    return Command(goto=goto)


def research_agent(state: MessagesState) -> Command:
    answer = llm.invoke(
        f"You are a research specialist. Briefly answer: {state['messages'][-1].content}"
    ).content
    from langchain_core.messages import AIMessage
    return Command(goto="supervisor", update={"messages": [AIMessage(content=f"[research_agent] {answer}")]})


def math_agent(state: MessagesState) -> Command:
    answer = llm.invoke(
        f"You are a math specialist. Solve, showing the final number clearly: {state['messages'][-1].content}"
    ).content
    from langchain_core.messages import AIMessage
    return Command(goto="supervisor", update={"messages": [AIMessage(content=f"[math_agent] {answer}")]})


supervisor_builder = StateGraph(MessagesState)
supervisor_builder.add_node("supervisor", supervisor)
supervisor_builder.add_node("research_agent", research_agent)
supervisor_builder.add_node("math_agent", math_agent)
supervisor_builder.add_edge(START, "supervisor")
supervisor_app = supervisor_builder.compile()

print(supervisor_app.get_graph().draw_mermaid())

for q in ["What is 47 * 13?", "What year was the Transformer paper published?"]:
    out = supervisor_app.invoke({"messages": [("user", q)]})
    print(q, "->", out["messages"][-1].content)

**Note on nodes that return `Command(goto=...)` directly:** you don't need
`add_conditional_edges` for these transitions — the routing decision travels *with* the
return value. This is exactly how the batteries-included
[`langgraph-supervisor`](https://github.com/langchain-ai/langgraph-supervisor-py) package
(`create_supervisor(...)`) builds the same pattern as a one-liner once you understand what it
compiles to.

## Summary

| Primitive | Gives you |
|---|---|
| `interrupt()` + `Command(resume=...)` | A graph that pauses, waits for a human, and resumes from the *exact* node that paused — not from the top |
| `Store` | Memory that survives across different `thread_id`s, not just within one |
| `Command(goto=...)` | A node that decides, at runtime, which node runs next — the basis of the supervisor pattern |

**Next: `03_agentic_rag_capstone.ipynb`** — all three of these, plus everything Module 10
built (hybrid retrieval, reranking, groundedness), combined into one self-correcting Agentic
RAG system.